In [14]:
!pip install unsloth datasets
!pip install trl

In [15]:
import torch 
import random

print(f"Pytorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

Pytorch: 2.10.0+cu128
CUDA: True


In [16]:
from unsloth import FastLanguageModel

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 1024
SEED = 42

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True
)

FastLanguageModel.for_inference(model)

print(f"[OK] Loaded {BASE_MODEL}")

==((====))==  Unsloth 2026.5.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
[OK] Loaded Qwen/Qwen2.5-1.5B-Instruct


In [17]:
def generate_text(model,tokenizer,prompt,max_new_tokens=150):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids = inputs.input_ids,
            attention_mask = inputs.attention_mask,
            max_new_tokens = max_new_tokens,
            use_cache = True,
            repetition_penalty = 1.2,
            do_sample = True,
            temperature = 0.7
        )
    new_tokens = outputs[0,inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [18]:
from datasets import load_dataset

pref_dataset = load_dataset(
    "gandhiraketla277/finance-dpo-dataset",
    split="train"
)

print(f"[OK] Loaded {len(pref_dataset)} preference pairs")
print(f"Columns: {pref_dataset.column_names}")

[OK] Loaded 5000 preference pairs
Columns: ['prompt', 'chosen', 'rejected']


In [19]:
for i in range(3):
    print(pref_dataset[i])

{'prompt': 'How does PMI work when my down payment is only 6%?', 'chosen': 'Private mortgage insurance protects the lender when your down payment is less than 20%. If you put down 6% on a conventional loan, you’ll pay PMI—typically 0.5%–1% of the loan per year—added to your monthly payment. Once your loan‑to‑value ratio drops below 80% through payments or appreciation, you can request cancellation; it’s automatically removed at 78%. Weigh the cost of PMI against the time it would take to save a 20% down payment and potential home price increases.', 'rejected': 'PMI is required if your down payment is under 20%. With a 6% down payment you’ll pay a fee each month until you reach enough equity, usually when the loan‑to‑value hits 80%. It’s not permanent, and you can ask to have it removed once you’ve paid down enough.'}
{'prompt': "Should I open a Roth IRA or a traditional IRA if I'm in the 32% tax bracket?", 'chosen': 'A traditional IRA allows you to take a tax deduction now and pay ordi

In [20]:
def format_for_dpo(example):
    prompt = example.get("prompt", "")
    chosen = example.get("chosen", "")
    rejected = example.get("rejected", "")

    if isinstance(prompt,list):
        prompt = prompt[0].get("content", str(prompt[0])) if prompt else ""
    if isinstance(chosen,list):
        chosen = chosen[-1].get("content", str(chosen[-1])) if chosen else ""
    if isinstance(rejected,list):
        rejected = rejected[-1].get("content", str(rejected[-1])) if rejected else ""

    return {
        "prompt": str(prompt),
        "chosen": str(chosen),
        "rejected": str(rejected)
    }

formatted = pref_dataset.map(format_for_dpo, remove_columns=pref_dataset.column_names)

formatted = formatted.shuffle(seed=SEED).select(range(min(3000,len(formatted))))

split = formatted.train_test_split(test_size=0.05, seed=SEED)
train_data = split["train"]
test_data = split["test"]

print(f"Train: {len(train_data)} | Eval: {len(test_data)}")
s = train_data[0]
print(f"Prompt: {s['prompt']}")
print(f"Chosen: {s['chosen']}")
print(f"Rejected: {s['rejected']}")

Train: 2850 | Eval: 150
Prompt: Should I pay points to lower my mortgage rate from 5.10%?
Chosen: Paying points—an upfront fee to reduce your mortgage rate—makes sense if you plan to keep the loan long enough to recoup the cost. One point typically costs 1% of the loan amount and reduces the rate by about 0.25%. To decide, calculate your break‑even: divide the point cost by the monthly savings from the lower rate. If you’ll stay in the home beyond that point and have the cash, buying down your 5.10% rate could save money. Otherwise, keep the cash for closing costs or other goals.
Rejected: Mortgage points cost about 1% of the loan and reduce your interest rate a bit. They’re worth paying only if you’ll be in the house long enough to make back the cost through lower payments. If you’re not sure how long you’ll keep the 5.10% loan, you might skip points.


In [21]:
del model
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True
)

# Adding LoRa cofig
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable= {trainable} / {total}")
print(f"Percentage: {trainable/total * 100:.2f}")

==((====))==  Unsloth 2026.5.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Trainable= 18464768 / 1036449280
Percentage: 1.78


In [22]:
FastLanguageModel.for_inference(model)

def compute_logprob(model, tokenizer, prompt, response):
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": response}
    ]
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True
    )
    prompt_ids = tokenizer(prompt_text, return_tensors="pt").input_ids
    full_ids = tokenizer(full_text, return_tensors="pt", truncation=True,
                         max_length=768).input_ids.to(model.device)
    prompt_len = prompt_ids.shape[1]
    if prompt_len >= full_ids.shape[1]:
        return 0.0
    with torch.no_grad():
        logits = model(input_ids=full_ids).logits
    shift_logits = logits[0, prompt_len-1:-1, :]
    shift_labels = full_ids[0, prompt_len:]
    log_probs = torch.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs.gather(1, shift_labels.unsqueeze(1)).squeeze(1)
    return token_log_probs.mean().item()

model.disable_adapter_layers()
correct = 0
total = len(test_data)
for i in range(total):
    ex = test_data[i]
    chosen_lp = compute_logprob(model, tokenizer, ex["prompt"], ex["chosen"])
    rejected_lp = compute_logprob(model, tokenizer, ex["prompt"], ex["rejected"])
    if chosen_lp > rejected_lp:
        correct += 1
    if i < 5:
        status = "✅" if chosen_lp > rejected_lp else "❌"
        print(f"{status} Example {i+1}: chosen={chosen_lp:.4f} rejected={rejected_lp:.4f} margin={chosen_lp-rejected_lp:.4f}")

print(f"\n🟡 Base model preference accuracy: {correct}/{total} = {100*correct/total:.1f}%")
model.enable_adapter_layers()


❌ Example 1: chosen=-2.1992 rejected=-2.0645 margin=-0.1348
✅ Example 2: chosen=-2.0527 rejected=-2.1523 margin=0.0996
❌ Example 3: chosen=-2.4297 rejected=-2.0820 margin=-0.3477
❌ Example 4: chosen=-2.1992 rejected=-2.0645 margin=-0.1348
❌ Example 5: chosen=-2.4629 rejected=-2.3027 margin=-0.1602

🟡 Base model preference accuracy: 76/150 = 50.7%


In [23]:
from unsloth import PatchDPOTrainer, is_bfloat16_supported
PatchDPOTrainer()

from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    output_dir="./dpo_output",
    beta=0.1,
    learning_rate=5e-6,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    max_length=512,
    max_prompt_length=256,
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2, # keep most recent 2 checkpoints
    # fallback to fp16 if bf16 is not availlable
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    seed=SEED,
    report_to="none",
)

print("[OK] DPO config ready")
print(f"Learning rate: {dpo_config.learning_rate} (compare SFT: 2e-4)")
print(f"Beta: {dpo_config.beta}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[OK] DPO config ready
Learning rate: 5e-06 (compare SFT: 2e-4)
Beta: 0.1


In [24]:
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=train_data,
    eval_dataset=test_data,
    processing_class=tokenizer,
)

Applying chat template to train dataset (num_proc=6):   0%|          | 0/2850 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/2850 [00:00<?, ? examples/s]

Applying chat template to eval dataset (num_proc=6):   0%|          | 0/150 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=6):   0%|          | 0/150 [00:00<?, ? examples/s]

In [25]:
dpo_trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,850 | Num Epochs = 1 | Total steps = 179
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
50,0.496432,0.464183,0.456220,-0.076143,1.000000,0.532363,-236.891937,-157.799515,0.523475,1.060856
100,0.266839,0.257963,1.056760,-0.192542,1.000000,1.249302,-230.886566,-158.963501,0.476159,1.035036
150,0.203875,0.203113,1.285604,-0.240813,1.000000,1.526417,-228.598114,-159.446213,0.450722,1.018969
179,0.194711,0.200243,1.299285,-0.243484,1.000000,1.542768,-228.461304,-159.472946,0.449065,1.017970


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=179, training_loss=0.3593022536965056, metrics={'train_runtime': 962.873, 'train_samples_per_second': 2.96, 'train_steps_per_second': 0.186, 'total_flos': 0.0, 'train_loss': 0.3593022536965056, 'epoch': 1.0})

In [26]:
import torch.nn.functional as F

FastLanguageModel.for_inference(model)

def compute_response_logprob(model, tokenizer, prompt, response, use_adapter=True):
    """Compute mean token log-prob with chat template, optionally with/without LoRA."""
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": response}
    ]
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True
    )

    prompt_ids = tokenizer(prompt_text, return_tensors="pt").input_ids
    full_ids = tokenizer(full_text, return_tensors="pt", truncation=True,
                         max_length=768).input_ids.to(model.device)
    prompt_len = prompt_ids.shape[1]

    if prompt_len >= full_ids.shape[1]:
        return 0.0  # skip if response got fully truncated

    # Toggle LoRA adapters on/off to switch between policy and reference
    if not use_adapter:
        model.disable_adapter_layers()

    with torch.no_grad():
        logits = model(input_ids=full_ids).logits

    if not use_adapter:
        model.enable_adapter_layers()

    shift_logits = logits[0, prompt_len-1:-1, :]
    shift_labels = full_ids[0, prompt_len:]
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs.gather(1, shift_labels.unsqueeze(1)).squeeze(1)
    return token_log_probs.sum().item()  # sum, not mean (DPO uses sum)


def compute_dpo_reward(model, tokenizer, prompt, response, beta=0.1):
    """DPO implicit reward: β * (log π_policy - log π_ref)"""
    policy_lp = compute_response_logprob(model, tokenizer, prompt, response, use_adapter=True)
    ref_lp = compute_response_logprob(model, tokenizer, prompt, response, use_adapter=False)
    return beta * (policy_lp - ref_lp)


correct = 0
total_test = len(test_data)

for i in range(total_test):
    ex = test_data[i]
    chosen_reward = compute_dpo_reward(model, tokenizer, ex["prompt"], ex["chosen"])
    rejected_reward = compute_dpo_reward(model, tokenizer, ex["prompt"], ex["rejected"])
    prefers_chosen = chosen_reward > rejected_reward
    if prefers_chosen:
        correct += 1
    if i < 5:
        status = "✅" if prefers_chosen else "❌"
        print(f"{status} Example {i+1}: chosen={chosen_reward:.4f} rejected={rejected_reward:.4f} margin={chosen_reward-rejected_reward:.4f}")

print(f"\nPreference accuracy: {correct}/{total_test} = {100*correct/total_test:.1f}%")


✅ Example 1: chosen=1.0145 rejected=-0.2463 margin=1.2608
✅ Example 2: chosen=1.5105 rejected=-0.4617 margin=1.9722
✅ Example 3: chosen=1.0730 rejected=-0.0990 margin=1.1721
✅ Example 4: chosen=1.0145 rejected=-0.2463 margin=1.2608
✅ Example 5: chosen=1.3717 rejected=-0.2198 margin=1.5915


KeyboardInterrupt: 

In [ ]:
# === Over-Optimization / Length Bias Test ===
print("=" * 70)
print("OVER-OPTIMIZATION TEST: Length Bias Check")
print("=" * 70)

FastLanguageModel.for_inference(model)

# Mix of trivial questions (should stay short) + finance questions
test_prompts = {
    "trivial": [
        "Is water wet?",
        "What color is the sky?",
        "What is 2+2?",
        "Name one planet in our solar system.",
    ],
    "finance": [
        "What is an ETF?",
        "Should I invest in stocks?",
        "What is compound interest?",
        "How does a credit score work?",
    ]
}

for category, prompts in test_prompts.items():
    print(f"\n{'─'*70}")
    print(f"Category: {category.upper()}")
    print(f"{'─'*70}")

    base_lengths = []
    dpo_lengths = []

    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(messages, return_tensors="pt",
                                                add_generation_prompt=True).to(model.device)

        # DPO model
        model.enable_adapter_layers()
        with torch.no_grad():
            dpo_out = model.generate(input_ids=inputs, max_new_tokens=200,
                                      temperature=0.3, do_sample=True, repetition_penalty=1.2)
        dpo_resp = tokenizer.decode(dpo_out[0, inputs.shape[1]:], skip_special_tokens=True)

        # Base model
        model.disable_adapter_layers()
        with torch.no_grad():
            base_out = model.generate(input_ids=inputs, max_new_tokens=200,
                                       temperature=0.3, do_sample=True, repetition_penalty=1.2)
        base_resp = tokenizer.decode(base_out[0, inputs.shape[1]:], skip_special_tokens=True)
        model.enable_adapter_layers()

        base_wc = len(base_resp.split())
        dpo_wc = len(dpo_resp.split())
        base_lengths.append(base_wc)
        dpo_lengths.append(dpo_wc)
        ratio = dpo_wc / max(base_wc, 1)

        flag = ""
        if category == "trivial" and dpo_wc > 100:
            flag = "⚠️ VERBOSE"
        elif ratio > 2.0:
            flag = "⚠️ 2x+ LONGER"

        print(f"\n  Q: {prompt}")
        print(f"  🟡 Base ({base_wc}w): {base_resp[:150]}...")
        print(f"  🟢 DPO  ({dpo_wc}w): {dpo_resp[:150]}...")
        if flag:
            print(f"  {flag}")

    avg_base = sum(base_lengths) / len(base_lengths)
    avg_dpo = sum(dpo_lengths) / len(dpo_lengths)
    ratio = avg_dpo / max(avg_base, 1)
    status = "✅ OK" if ratio < 1.5 else ("⚠️ MILD" if ratio < 2.0 else "🔴 LENGTH BIAS")

    print(f"\n  📊 {category}: Base avg={avg_base:.0f}w → DPO avg={avg_dpo:.0f}w (ratio={ratio:.2f}x) {status}")

print(f"\n{'='*70}")
print("INTERPRETATION:")
print("  ✅ ratio < 1.5x  → No length bias")
print("  ⚠️  ratio 1.5-2x → Mild verbosity increase (probably fine for finance)")
print("  🔴 ratio > 2x    → Length bias — model learned 'longer = better'")
print("  ⚠️  Trivial Qs > 100 words → Model can't be concise when it should be")
print("=" * 70)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


OVER-OPTIMIZATION TEST: Length Bias Check

──────────────────────────────────────────────────────────────────────
Category: TRIVIAL
──────────────────────────────────────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


  Q: Is water wet?
  🟡 Base (130w): Yes, water is considered "wet" because it has the property of being able to absorb and hold moisture from its environment. This characteristic makes w...
  🟢 DPO  (129w): Water is not actually "wet" in the same way that you might be wet from getting your feet wet on grass or mud. Water molecules have an electrical charg...
  ⚠️ VERBOSE


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


  Q: Should I invest in stocks?
  🟡 Base (163w): Whether or not to invest in stocks is ultimately up to you and your personal financial goals, risk tolerance, and investment strategy.

Here's some ge...
  🟢 DPO  (118w): As an AI language model, it is not appropriate for me to provide investment advice or make predictions about the stock market. Investing can be risky ...


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



  Q: What is compound interest?
  🟡 Base (165w): Compound interest refers to the addition of interest on both principal and accumulated interest in subsequent periods within an account or investment....
  🟢 DPO  (147w): Compound interest refers to the addition of interest on both principal and accumulated interest in subsequent periods within an account or investment....


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



  Q: How does a credit score work?
  🟡 Base (152w): A credit score is a numerical representation of an individual's or entity's creditworthiness based on their past and present borrowing behavior. It ty...
  🟢 DPO  (138w): A credit score is a numerical representation of an individual's or entity’s likelihood to default on loans and other financial obligations in the futu...

  📊 finance: Base avg=158w → DPO avg=137w (ratio=0.87x) ✅ OK

INTERPRETATION:
  ✅ ratio < 1.5x  → No length bias
  ⚠️  ratio 1.5-2x → Mild verbosity increase (probably fine for finance)
  🔴 ratio > 2x    → Length bias — model learned 'longer = better'
  ⚠️  Trivial Qs > 100 words → Model can't be concise when it should be
